In [ ]:
from pathlib import Path
Path('/kaggle/working/v3_b3_source').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/v3_b3_source/efficientnet_b3_public_repro_v1.py
#!/usr/bin/env python3
"""Leakage-controlled EfficientNet-B3 MIL training for RSNA Knee."""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import re
import unicodedata
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

TARGETS: Tuple[str, ...] = (
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
)
PLANES: Tuple[str, ...] = ("Sagittal", "Coronal", "Axial")
UID = "StudyInstanceUID"


@dataclass(frozen=True)
class TrainConfig:
    image_size: int = 288
    max_series_slices: int = 64
    train_slices: int = 20
    valid_slices: int = 32
    folds: int = 5
    seed: int = 20260809
    batch_size: int = 2
    workers: int = 4
    backbone_chunk: int = 12
    frozen_epochs: int = 1
    unfrozen_epochs: int = 2
    frozen_lr: float = 8e-4
    unfrozen_lr: float = 8e-5
    weight_decay: float = 1e-4
    gold_weight: float = 3.0
    pseudo_weight_floor: float = 0.20
    backbone: str = "efficientnet_b3"


_DATE = re.compile(
    r"\b(?:19|20)\d{2}[-/.]\d{1,2}[-/.]\d{1,2}\b|"
    r"\b\d{1,2}[-/.]\d{1,2}[-/.](?:\d{2}|\d{4})\b"
)
_NUMBER = re.compile(r"\b\d+(?:[.,]\d+)?\b")
_SPACE = re.compile(r"\s+")


def normalize_report_for_group(value: object) -> str:
    text = "" if value is None else str(value)
    text = unicodedata.normalize("NFKC", text).casefold()
    text = _DATE.sub(" <date> ", text)
    text = _NUMBER.sub(" <num> ", text)
    return _SPACE.sub(" ", text).strip()


def report_group_id(value: object) -> str:
    return hashlib.sha256(normalize_report_for_group(value).encode()).hexdigest()[:24]


def _indexed_labels(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    required = {UID, *TARGETS}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")
    if frame[UID].duplicated().any():
        raise ValueError(f"{name} contains duplicate {UID} rows")
    return frame[[UID, *TARGETS]].copy().set_index(UID).apply(pd.to_numeric, errors="coerce")


def build_public_consensus(
    train: pd.DataFrame,
    pilkwang: pd.DataFrame,
    steven: pd.DataFrame,
    lixin: pd.DataFrame,
    pseudo_weight_floor: float = 0.20,
) -> pd.DataFrame:
    """Equal-source soft labels; weights come only from agreement/certainty."""
    if train[UID].duplicated().any():
        raise ValueError(f"train contains duplicate {UID} rows")
    base = train[[UID, "Report", *TARGETS]].copy().set_index(UID)
    sources = [
        _indexed_labels(pilkwang, "pilkwang"),
        _indexed_labels(steven, "steven"),
        _indexed_labels(lixin, "lixin"),
    ]
    out = base.reset_index()[[UID, "Report"]].copy()
    out["report_group"] = out["Report"].map(report_group_id)
    out = out.drop(columns="Report")
    cube = np.stack([source.reindex(base.index).to_numpy(float) for source in sources])
    if np.nanmin(cube) < 0.0 or np.nanmax(cube) > 1.0:
        raise ValueError("public labels must be in [0, 1]")
    available = np.isfinite(cube).sum(axis=0)
    if np.any(available < 2):
        row, target = np.argwhere(available < 2)[0]
        raise ValueError(f"fewer than two public labels at row={row}, target={TARGETS[target]}")
    consensus = np.nanmean(cube, axis=0)
    disagreement = np.nanmean(np.abs(cube - consensus[None]), axis=0)
    agreement = np.clip(1.0 - 2.0 * disagreement, 0.0, 1.0)
    certainty = np.clip(2.0 * np.abs(consensus - 0.5), 0.0, 1.0)
    weight = pseudo_weight_floor + (1.0 - pseudo_weight_floor) * (
        0.65 * agreement + 0.35 * certainty
    )
    out["is_expert"] = base[list(TARGETS)].notna().all(axis=1).astype(np.int8).to_numpy()
    for j, target in enumerate(TARGETS):
        out[f"pseudo::{target}"] = consensus[:, j].astype(np.float32)
        out[f"weight::{target}"] = weight[:, j].astype(np.float32)
        out[f"gold::{target}"] = pd.to_numeric(base[target], errors="coerce").to_numpy()
    return out


def _group_statistics(label_table: pd.DataFrame) -> pd.DataFrame:
    pseudo_cols = [f"pseudo::{target}" for target in TARGETS]
    gold_cols = [f"gold::{target}" for target in TARGETS]
    records = []
    for group_id, group in label_table.groupby("report_group", sort=True):
        gold = group[gold_cols].to_numpy(float)
        records.append({
            "report_group": group_id,
            "size": len(group),
            "expert_count": int(group["is_expert"].sum()),
            "pseudo_sum": group[pseudo_cols].to_numpy(float).sum(axis=0),
            "gold_positive": np.nansum(gold, axis=0),
            "gold_known": np.isfinite(gold).sum(axis=0).astype(float),
        })
    return pd.DataFrame(records)


def assign_group_balanced_folds(
    label_table: pd.DataFrame, n_splits: int = 5, seed: int = 20260809
) -> pd.Series:
    """Deterministic greedy balance with normalized-report groups kept atomic."""
    if n_splits < 2:
        raise ValueError("n_splits must be at least two")
    groups = _group_statistics(label_table)
    target_size = float(groups["size"].sum()) / n_splits
    target_pseudo = np.sum(np.stack(groups["pseudo_sum"]), axis=0) / n_splits
    total_gold_pos = np.sum(np.stack(groups["gold_positive"]), axis=0)
    target_gold_pos = total_gold_pos / n_splits
    target_gold_known = np.sum(np.stack(groups["gold_known"]), axis=0) / n_splits
    rng = np.random.default_rng(seed)
    rarity = 1.0 / np.maximum(total_gold_pos, 1.0)
    priority = [
        100.0 * row["expert_count"]
        + 10.0 * float(np.dot(row["gold_positive"], rarity))
        + math.log1p(row["size"])
        for _, row in groups.iterrows()
    ]
    groups = groups.assign(_priority=priority, _jitter=rng.uniform(0, 1e-6, len(groups)))
    groups = groups.sort_values(["_priority", "_jitter", "report_group"], ascending=[False, False, True])
    fold_size = np.zeros(n_splits)
    fold_pseudo = np.zeros((n_splits, len(TARGETS)))
    fold_gold_pos = np.zeros_like(fold_pseudo)
    fold_gold_known = np.zeros_like(fold_pseudo)
    assignment: Dict[str, int] = {}
    for _, row in groups.iterrows():
        scores = []
        for fold in range(n_splits):
            candidate_size = fold_size.copy()
            candidate_pseudo = fold_pseudo.copy()
            candidate_gold_pos = fold_gold_pos.copy()
            candidate_gold_known = fold_gold_known.copy()
            candidate_size[fold] += row["size"]
            candidate_pseudo[fold] += row["pseudo_sum"]
            candidate_gold_pos[fold] += row["gold_positive"]
            candidate_gold_known[fold] += row["gold_known"]
            size_cost = np.mean(((candidate_size - target_size) / max(target_size, 1.0)) ** 2)
            pseudo_cost = np.mean(((candidate_pseudo - target_pseudo[None]) / np.maximum(target_pseudo[None], 1.0)) ** 2)
            gold_pos_cost = np.mean(((candidate_gold_pos - target_gold_pos[None]) / np.maximum(target_gold_pos[None], 1.0)) ** 2)
            gold_known_cost = np.mean(((candidate_gold_known - target_gold_known[None]) / np.maximum(target_gold_known[None], 1.0)) ** 2)
            scores.append(size_cost + 0.35 * pseudo_cost + 4.0 * gold_pos_cost + 2.0 * gold_known_cost)
        best = int(np.argmin(np.asarray(scores) + 1e-12 * np.arange(n_splits)))
        assignment[str(row["report_group"])] = best
        fold_size[best] += row["size"]
        fold_pseudo[best] += row["pseudo_sum"]
        fold_gold_pos[best] += row["gold_positive"]
        fold_gold_known[best] += row["gold_known"]
    result = label_table["report_group"].map(assignment)
    if result.isna().any():
        raise AssertionError("some report groups were not assigned")
    return result.astype(np.int8)


def pick_one_series_per_plane(series: pd.DataFrame) -> Dict[str, Optional[str]]:
    required = {"SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"}
    missing = required.difference(series.columns)
    if missing:
        raise ValueError(f"series table missing columns: {sorted(missing)}")
    chosen: Dict[str, Optional[str]] = {}
    for plane in PLANES:
        subset = series.loc[series["Anatomical_Plane"].eq(plane)].copy()
        if subset.empty:
            chosen[plane] = None
            continue
        for column in ("Fluid_Sensitive", "Fat_Suppression"):
            subset[column] = pd.to_numeric(subset[column], errors="coerce").fillna(0)
        subset = subset.sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
            ascending=[False, False, True],
        )
        chosen[plane] = str(subset.iloc[0]["SeriesInstanceUID"])
    return chosen


def pick_series_slots(series: pd.DataFrame, count: int = 3) -> List[Optional[str]]:
    """Prefer one fluid-sensitive series per plane, then fill empty slots."""
    required = {"SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"}
    missing = required.difference(series.columns)
    if missing:
        raise ValueError(f"series table missing columns: {sorted(missing)}")
    frame = series.copy()
    for column in ("Fluid_Sensitive", "Fat_Suppression"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0)
    frame["_plane"] = frame["Anatomical_Plane"].map({p: i for i, p in enumerate(PLANES)}).fillna(3)
    preferred = frame.loc[frame["Fluid_Sensitive"].eq(1)].sort_values(
        ["_plane", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[True, False, True],
    )
    selected: List[str] = []
    seen_planes = set()
    for _, row in preferred.iterrows():
        plane = str(row["Anatomical_Plane"])
        if plane not in seen_planes:
            selected.append(str(row["SeriesInstanceUID"]))
            seen_planes.add(plane)
        if len(selected) == count:
            break
    ordered = frame.sort_values(
        ["Fluid_Sensitive", "Fat_Suppression", "_plane", "SeriesInstanceUID"],
        ascending=[False, False, True, True],
    )
    for series_uid in ordered["SeriesInstanceUID"].astype(str):
        if len(selected) == count:
            break
        if series_uid not in selected:
            selected.append(series_uid)
    return selected + [None] * (count - len(selected))


def uniform_slice_indices(length: int, count: int) -> np.ndarray:
    if length <= 0 or count <= 0:
        raise ValueError("length and count must be positive")
    if length <= count:
        return np.concatenate([np.arange(length), np.full(count - length, length - 1)]).astype(np.int64)
    return np.rint(np.linspace(0, length - 1, count)).astype(np.int64)


def stochastic_slice_indices(length: int, count: int, rng: np.random.Generator) -> np.ndarray:
    if length <= count:
        return uniform_slice_indices(length, count)
    edges = np.linspace(0, length, count + 1)
    values = []
    for left, right in zip(edges[:-1], edges[1:]):
        lo = int(math.floor(left))
        hi = max(lo + 1, int(math.ceil(right)))
        values.append(int(rng.integers(lo, min(hi, length))))
    return np.asarray(values, dtype=np.int64)


def pooled_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> Tuple[float, Dict[str, float]]:
    from sklearn.metrics import roc_auc_score
    per_target: Dict[str, float] = {}
    for j, target in enumerate(TARGETS):
        valid = np.isfinite(y_true[:, j]) & np.isfinite(y_score[:, j])
        per_target[target] = (
            float(roc_auc_score(y_true[valid, j], y_score[valid, j]))
            if valid.sum() and np.unique(y_true[valid, j]).size == 2
            else float("nan")
        )
    finite = [value for value in per_target.values() if np.isfinite(value)]
    return (float(np.mean(finite)) if finite else float("nan")), per_target


def prepare_fold_rows(
    label_table: pd.DataFrame, fold: int, config: TrainConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_rows = label_table.loc[label_table["fold"].ne(fold)].copy()
    valid_rows = label_table.loc[label_table["fold"].eq(fold) & label_table["is_expert"].eq(1)].copy()
    if train_rows.empty or valid_rows.empty:
        raise ValueError(f"empty train/validation split for fold {fold}")
    for target in TARGETS:
        gold = pd.to_numeric(train_rows[f"gold::{target}"], errors="coerce")
        is_gold = train_rows["is_expert"].eq(1) & gold.notna()
        train_rows[f"target::{target}"] = train_rows[f"pseudo::{target}"].where(~is_gold, gold)
        train_rows[f"train_weight::{target}"] = train_rows[f"weight::{target}"].where(~is_gold, config.gold_weight)
        valid_rows[f"target::{target}"] = pd.to_numeric(valid_rows[f"gold::{target}"], errors="raise")
        valid_rows[f"train_weight::{target}"] = 1.0
    if set(train_rows["report_group"]).intersection(valid_rows["report_group"]):
        raise AssertionError("normalized report leakage across train and validation")
    return train_rows, valid_rows


def _training_imports():
    try:
        import cv2
        import pydicom
        import timm
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, Dataset
    except ImportError as exc:
        raise RuntimeError("training requires torch, timm, pydicom, and opencv-python") from exc
    return cv2, pydicom, timm, torch, nn, DataLoader, Dataset


def load_dicom_volume(series_dir: Path, image_size: int, max_slices: int = 64) -> np.ndarray:
    cv2, pydicom, _, _, _, _, _ = _training_imports()
    slices: List[Tuple[int, np.ndarray]] = []
    for path in sorted(series_dir.glob("*.dcm")):
        try:
            ds = pydicom.dcmread(str(path))
            image = ds.pixel_array.astype(np.float32)
            image = image * float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
            image += float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
            if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
                image = image.max() - image
            slices.append((int(getattr(ds, "InstanceNumber", 0) or 0), image))
        except Exception:
            continue
    if not slices:
        raise RuntimeError(f"no decodable slices in {series_dir}")
    slices.sort(key=lambda item: item[0])
    volume = np.stack([item[1] for item in slices])
    lo, hi = np.percentile(volume, [1.0, 99.0])
    hi = max(float(hi), float(lo) + 1.0)
    volume = np.clip((volume - lo) / (hi - lo), 0.0, 1.0)
    if len(volume) > max_slices:
        start = (len(volume) - max_slices) // 2
        volume = volume[start:start + max_slices]
    resized = np.stack([
        cv2.resize(x, (image_size, image_size), interpolation=cv2.INTER_AREA)
        for x in volume
    ])
    # Match the public .903 pipeline's training-time uint8 ETL boundary.
    return (resized * 255.0).astype(np.uint8).astype(np.float32) / 255.0


def make_dataset_class():
    _, _, _, torch, _, _, Dataset = _training_imports()
    class KneeStudyDataset(Dataset):
        def __init__(self, rows, series, image_root, config, training, seed):
            self.rows = rows.reset_index(drop=True)
            self.series = {str(uid): group.copy() for uid, group in series.groupby(UID, sort=False)}
            self.image_root, self.config, self.training, self.seed = Path(image_root), config, training, seed

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, index):
            row, config = self.rows.iloc[index], self.config
            uid = str(row[UID])
            count = config.train_slices if self.training else config.valid_slices
            # DataLoader seeds each worker deterministically.  Drawing once from
            # its advancing torch RNG gives a fresh augmentation on every visit
            # without sacrificing run-level reproducibility.
            visit_seed = int(torch.randint(0, 2**31 - 1, (1,)).item()) if self.training else 0
            rng = np.random.default_rng(self.seed + 1_000_003 * index + visit_seed)
            empty = pd.DataFrame(columns=["SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"])
            chosen = pick_series_slots(self.series.get(uid, empty), len(PLANES))
            planes, masks = [], []
            for series_uid in chosen:
                if series_uid is None:
                    planes.append(np.zeros((count, config.image_size, config.image_size), np.float32))
                    masks.append(np.zeros(count, bool))
                    continue
                volume = load_dicom_volume(
                    self.image_root / uid / series_uid,
                    config.image_size,
                    config.max_series_slices,
                )
                idx = stochastic_slice_indices(len(volume), count, rng) if self.training else uniform_slice_indices(len(volume), count)
                sampled = volume[idx].astype(np.float32)
                if self.training:
                    sampled = np.clip(sampled * rng.uniform(0.85, 1.15), 0, 1) ** rng.uniform(0.85, 1.15)
                    if rng.random() < 0.5:
                        sampled = sampled[:, :, ::-1].copy()
                sampled = (sampled - 0.449) / 0.226
                planes.append(sampled)
                masks.append(np.ones(count, bool))
            return {
                "uid": uid,
                "images": torch.from_numpy(np.stack(planes)[:, :, None]),
                "mask": torch.from_numpy(np.stack(masks)),
                "target": torch.tensor([row[f"target::{x}"] for x in TARGETS], dtype=torch.float32),
                "weight": torch.tensor([row[f"train_weight::{x}"] for x in TARGETS], dtype=torch.float32),
            }
    return KneeStudyDataset


def make_model_class():
    _, _, timm, torch, nn, _, _ = _training_imports()
    class KneeMILModel(nn.Module):
        def __init__(self, backbone="efficientnet_b3", checkpoint=None, backbone_chunk=12):
            super().__init__()
            self.backbone_chunk = int(backbone_chunk)
            self.backbone = timm.create_model(backbone, pretrained=False, in_chans=1, num_classes=0, global_pool="avg")
            if checkpoint:
                state = torch.load(str(checkpoint), map_location="cpu", weights_only=False)
                if isinstance(state, Mapping) and "state_dict" in state:
                    state = state["state_dict"]
                state = {str(k).replace("module.", "", 1): v for k, v in state.items()}
                expected = self.backbone.state_dict()
                if "conv_stem.weight" in state and state["conv_stem.weight"].shape[1] == 3:
                    state["conv_stem.weight"] = state["conv_stem.weight"].sum(dim=1, keepdim=True)
                state = {key: value for key, value in state.items() if key in expected}
                shape_errors = {
                    key: (tuple(value.shape), tuple(expected[key].shape))
                    for key, value in state.items()
                    if value.shape != expected[key].shape
                }
                if shape_errors:
                    raise RuntimeError(f"backbone tensor shape mismatch: {shape_errors}")
                result = self.backbone.load_state_dict(state, strict=True)
                if result.missing_keys or result.unexpected_keys:
                    raise RuntimeError(f"backbone state mismatch: {result}")
            features, hidden = int(self.backbone.num_features), 384
            self.attn_v, self.attn_u = nn.Linear(features, hidden), nn.Linear(features, hidden)
            self.attn_out = nn.Linear(hidden, len(TARGETS))
            self.target_weight = nn.Parameter(torch.empty(len(TARGETS), features))
            self.target_bias = nn.Parameter(torch.zeros(len(TARGETS)))
            self.slice_head = nn.Linear(features, len(TARGETS))
            self.mix_logit = nn.Parameter(torch.tensor(-1.1))
            nn.init.xavier_uniform_(self.target_weight)

        def set_backbone_trainable(self, value):
            for parameter in self.backbone.parameters():
                parameter.requires_grad = value

        def forward(self, images, mask):
            batch, planes, slices, channels, height, width = images.shape
            flat = images.reshape(batch * planes * slices, channels, height, width)
            features = []
            train_backbone = self.training and any(p.requires_grad for p in self.backbone.parameters())
            for start in range(0, len(flat), self.backbone_chunk):
                chunk = flat[start:start + self.backbone_chunk]
                if train_backbone:
                    from torch.utils.checkpoint import checkpoint
                    features.append(checkpoint(self.backbone, chunk, use_reentrant=False))
                else:
                    features.append(self.backbone(chunk))
            features = torch.cat(features, dim=0)
            features = features.reshape(batch, planes * slices, -1)
            flat_mask = mask.reshape(batch, planes * slices)
            gated = torch.tanh(self.attn_v(features)) * torch.sigmoid(self.attn_u(features))
            attention = self.attn_out(gated).masked_fill(~flat_mask[:, :, None], -1e4).softmax(dim=1)
            pooled = torch.einsum("bnt,bnf->btf", attention, features)
            attention_logits = torch.einsum("btf,tf->bt", pooled, self.target_weight) + self.target_bias
            max_logits = self.slice_head(features).masked_fill(~flat_mask[:, :, None], -1e4).max(dim=1).values
            mix = torch.sigmoid(self.mix_logit)
            return (1.0 - mix) * attention_logits + mix * max_logits
    return KneeMILModel


def train_one_fold(label_table, series, image_root, output_dir, fold, backbone_checkpoint, config):
    _, _, _, torch, nn, DataLoader, _ = _training_imports()
    random.seed(config.seed + fold)
    np.random.seed(config.seed + fold)
    torch.manual_seed(config.seed + fold)
    Dataset, Model = make_dataset_class(), make_model_class()
    train_rows, valid_rows = prepare_fold_rows(label_table, fold, config)
    train_loader = DataLoader(Dataset(train_rows, series, image_root, config, True, config.seed + fold), batch_size=config.batch_size, shuffle=True, num_workers=config.workers, pin_memory=True, persistent_workers=config.workers > 0)
    valid_loader = DataLoader(Dataset(valid_rows, series, image_root, config, False, config.seed + fold), batch_size=1, shuffle=False, num_workers=max(1, config.workers // 2))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("GPU is required")
    core_model = Model(config.backbone, backbone_checkpoint, config.backbone_chunk).to(device)
    model = nn.DataParallel(core_model) if torch.cuda.device_count() > 1 else core_model
    loss_fn, scaler, history = nn.BCEWithLogitsLoss(reduction="none"), torch.amp.GradScaler("cuda"), []
    schedule = [(False, config.frozen_lr)] * config.frozen_epochs + [(True, config.unfrozen_lr)] * config.unfrozen_epochs
    optimizer, active_phase = None, None
    for epoch, (unfreeze, lr) in enumerate(schedule):
        if active_phase != unfreeze:
            core_model.set_backbone_trainable(unfreeze)
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=lr,
                weight_decay=config.weight_decay,
            )
            active_phase = unfreeze
        assert optimizer is not None
        model.train()
        for layer in core_model.backbone.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                layer.eval()
        losses = []
        for batch in train_loader:
            images, mask = batch["images"].to(device), batch["mask"].to(device)
            target, weight = batch["target"].to(device), batch["weight"].to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                raw = loss_fn(model(images, mask), target)
                loss = (raw * weight).sum() / weight.sum().clamp_min(1.0)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "backbone_trainable": unfreeze,
            "learning_rate": lr,
        })
    model.eval()
    uids, truth, score = [], [], []
    with torch.no_grad():
        for batch in valid_loader:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(batch["images"].to(device), batch["mask"].to(device))
            uids.extend(batch["uid"])
            truth.append(batch["target"].numpy())
            score.append(torch.sigmoid(logits.float()).cpu().numpy())
    truth, score = np.concatenate(truth), np.concatenate(score)
    macro, per_target = pooled_macro_auc(truth, score)
    output_dir.mkdir(parents=True, exist_ok=True)
    oof = pd.DataFrame({UID: uids})
    for j, target in enumerate(TARGETS):
        oof[f"gold::{target}"], oof[f"pred::{target}"] = truth[:, j], score[:, j]
    oof.to_csv(output_dir / f"fold{fold}_expert_oof.csv", index=False)
    destination = output_dir / f"fold{fold}_final.pt"
    torch.save({"model": core_model.state_dict(), "backbone": config.backbone, "config": asdict(config), "fold": fold, "fold_diagnostic_macro_auc": macro, "fold_diagnostic_per_target_auc": per_target, "history": history, "visible_gpus": torch.cuda.device_count(), "backbone_checkpoint_sha256": hashlib.sha256(backbone_checkpoint.read_bytes()).hexdigest()}, destination)
    return destination


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("train-csv", "series-csv", "pilkwang-labels", "steven-labels", "lixin-labels", "output-dir"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--image-root", type=Path)
    parser.add_argument("--backbone-checkpoint", type=Path)
    parser.add_argument("--fold", type=int)
    parser.add_argument("--prepare-only", action="store_true")
    args = parser.parse_args(argv)
    config = TrainConfig()
    paths = [args.train_csv, args.series_csv, args.pilkwang_labels, args.steven_labels, args.lixin_labels]
    frames = [pd.read_csv(path) for path in paths]
    labels = build_public_consensus(frames[0], frames[2], frames[3], frames[4], config.pseudo_weight_floor)
    labels["fold"] = assign_group_balanced_folds(labels, config.folds, config.seed)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    labels.to_csv(args.output_dir / "fold_labels.csv", index=False)
    manifest = {
        "config": asdict(config), "rows": len(labels), "expert_rows": int(labels["is_expert"].sum()),
        "fold_counts": labels["fold"].value_counts().sort_index().to_dict(),
        "fold_expert_counts": labels.loc[labels["is_expert"].eq(1), "fold"].value_counts().sort_index().to_dict(),
        "inputs": {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths},
    }
    (args.output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    print(json.dumps(manifest, indent=2, sort_keys=True))
    if args.prepare_only:
        return
    if args.fold is None or args.image_root is None or args.backbone_checkpoint is None:
        parser.error("training requires --fold, --image-root, and --backbone-checkpoint")
    if not 0 <= args.fold < config.folds:
        parser.error(f"--fold must be in [0, {config.folds - 1}]")
    print("saved", train_one_fold(labels, frames[1], args.image_root, args.output_dir, args.fold, args.backbone_checkpoint, config))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/v3_b3_source/efficientnet_b3_public_repro_v2_anatomy.py
#!/usr/bin/env python3
"""Anatomy-aware EfficientNet-B3 MIL training for RSNA Knee.

This module deliberately reuses the audited v1 DICOM and label pipeline while
changing only three trainable assumptions that v1 could not express:

* each slice feature receives its acquisition-plane and normalized stack position;
* expert-labelled studies are sampled often enough to influence every epoch;
* fine-tuning is restricted to the final two EfficientNet stages.

The output checkpoint contract remains compatible with the v1 inference runner,
provided this v2 module is supplied to ``--module`` at inference time.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import math
import random
import sys
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Mapping, Optional, Sequence

import numpy as np
import pandas as pd


def _load_base_module():
    path = Path(__file__).with_name("efficientnet_b3_public_repro_v1.py")
    if not path.is_file():
        raise FileNotFoundError(f"audited v1 dependency is absent: {path}")
    name = "rsna_knee_effnet_b3_public_repro_v1_dependency"
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import audited v1 dependency: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


BASE = _load_base_module()
TARGETS = BASE.TARGETS
PLANES = BASE.PLANES
UID = BASE.UID
TrainConfig = BASE.TrainConfig
load_dicom_volume = BASE.load_dicom_volume
uniform_slice_indices = BASE.uniform_slice_indices
stochastic_slice_indices = BASE.stochastic_slice_indices
pooled_macro_auc = BASE.pooled_macro_auc


@dataclass(frozen=True)
class TrainingPolicy:
    """Choices specific to v2 and therefore stored separately from TrainConfig."""

    expert_sample_fraction: float = 0.12
    unfrozen_backbone_blocks: int = 2
    gradient_clip_norm: float = 5.0
    model_family: str = "efficientnet_b3_anatomy_mil_v2"


def _training_imports():
    return BASE._training_imports()


def pick_series_slots(series: pd.DataFrame, count: int = 3):
    """Return one fixed slot per anatomical plane.

    The v1 selector intentionally backfills a missing plane with another useful
    series.  That is appropriate for an anatomy-agnostic model, but would attach
    the wrong plane embedding in v2.  Preserve empty slots instead so index zero,
    one, and two always mean sagittal, coronal, and axial respectively.
    """
    if int(count) != len(PLANES):
        raise ValueError(f"v2 requires exactly {len(PLANES)} anatomical plane slots")
    selected = BASE.pick_one_series_per_plane(series)
    return [selected[plane] for plane in PLANES]


def make_dataset_class():
    """Build the v1 study dataset with fixed anatomical plane slots."""
    _, _, _, torch, _, _, Dataset = _training_imports()

    class AnatomyAwareKneeStudyDataset(Dataset):
        def __init__(self, rows, series, image_root, config, training, seed):
            self.rows = rows.reset_index(drop=True)
            self.series = {
                str(uid): group.copy() for uid, group in series.groupby(UID, sort=False)
            }
            self.image_root = Path(image_root)
            self.config = config
            self.training = bool(training)
            self.seed = int(seed)

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, index):
            row, config = self.rows.iloc[index], self.config
            uid = str(row[UID])
            count = config.train_slices if self.training else config.valid_slices
            visit_seed = (
                int(torch.randint(0, 2**31 - 1, (1,)).item()) if self.training else 0
            )
            rng = np.random.default_rng(self.seed + 1_000_003 * index + visit_seed)
            empty = pd.DataFrame(
                columns=[
                    "SeriesInstanceUID",
                    "Anatomical_Plane",
                    "Fluid_Sensitive",
                    "Fat_Suppression",
                ]
            )
            chosen = pick_series_slots(self.series.get(uid, empty), len(PLANES))
            planes, masks = [], []
            for series_uid in chosen:
                if series_uid is None:
                    planes.append(
                        np.zeros((count, config.image_size, config.image_size), np.float32)
                    )
                    masks.append(np.zeros(count, bool))
                    continue
                volume = load_dicom_volume(
                    self.image_root / uid / series_uid,
                    config.image_size,
                    config.max_series_slices,
                )
                indices = (
                    stochastic_slice_indices(len(volume), count, rng)
                    if self.training
                    else uniform_slice_indices(len(volume), count)
                )
                sampled = volume[indices].astype(np.float32)
                if self.training:
                    sampled = np.clip(sampled * rng.uniform(0.85, 1.15), 0, 1)
                    sampled = sampled ** rng.uniform(0.85, 1.15)
                    if rng.random() < 0.5:
                        sampled = sampled[:, :, ::-1].copy()
                sampled = (sampled - 0.449) / 0.226
                planes.append(sampled)
                masks.append(np.ones(count, bool))
            return {
                "uid": uid,
                "images": torch.from_numpy(np.stack(planes)[:, :, None]),
                "mask": torch.from_numpy(np.stack(masks)),
                "target": torch.tensor(
                    [row[f"target::{target}"] for target in TARGETS], dtype=torch.float32
                ),
                "weight": torch.tensor(
                    [row[f"train_weight::{target}"] for target in TARGETS],
                    dtype=torch.float32,
                ),
            }

    return AnatomyAwareKneeStudyDataset


def make_model_class():
    """Build a target-attention MIL model with fixed anatomical coordinates."""
    _, _, timm, torch, nn, _, _ = _training_imports()

    class AnatomyAwareKneeMIL(nn.Module):
        def __init__(self, backbone="efficientnet_b3", checkpoint=None, backbone_chunk=12):
            super().__init__()
            self.backbone_chunk = int(backbone_chunk)
            self.backbone = timm.create_model(
                backbone,
                pretrained=False,
                in_chans=1,
                num_classes=0,
                global_pool="avg",
            )
            if checkpoint:
                state = torch.load(str(checkpoint), map_location="cpu", weights_only=False)
                if isinstance(state, Mapping) and "state_dict" in state:
                    state = state["state_dict"]
                state = {str(key).replace("module.", "", 1): value for key, value in state.items()}
                expected = self.backbone.state_dict()
                if "conv_stem.weight" in state and state["conv_stem.weight"].shape[1] == 3:
                    # Sum rather than mean preserves the response magnitude expected by
                    # the pretrained filters after the one-channel ImageNet normalization.
                    state["conv_stem.weight"] = state["conv_stem.weight"].sum(dim=1, keepdim=True)
                state = {key: value for key, value in state.items() if key in expected}
                shape_errors = {
                    key: (tuple(value.shape), tuple(expected[key].shape))
                    for key, value in state.items()
                    if value.shape != expected[key].shape
                }
                if shape_errors:
                    raise RuntimeError(f"backbone tensor shape mismatch: {shape_errors}")
                result = self.backbone.load_state_dict(state, strict=True)
                if result.missing_keys or result.unexpected_keys:
                    raise RuntimeError(f"backbone state mismatch: {result}")

            features, hidden = int(self.backbone.num_features), 384
            self.plane_embedding = nn.Embedding(len(PLANES), features)
            self.position_projection = nn.Linear(4, features, bias=False)
            nn.init.normal_(self.plane_embedding.weight, std=0.01)
            nn.init.normal_(self.position_projection.weight, std=0.01)

            self.attn_v = nn.Linear(features, hidden)
            self.attn_u = nn.Linear(features, hidden)
            self.attn_out = nn.Linear(hidden, len(TARGETS))
            self.target_weight = nn.Parameter(torch.empty(len(TARGETS), features))
            self.target_bias = nn.Parameter(torch.zeros(len(TARGETS)))
            self.slice_head = nn.Linear(features, len(TARGETS))
            self.mix_logit = nn.Parameter(torch.tensor(-1.1))
            nn.init.xavier_uniform_(self.target_weight)

        def set_backbone_stage(self, final_blocks=0):
            """Freeze the backbone, optionally reopening only its final stages."""
            for parameter in self.backbone.parameters():
                parameter.requires_grad_(False)
            final_blocks = int(final_blocks)
            if final_blocks <= 0:
                return
            blocks = list(self.backbone.blocks)
            if final_blocks > len(blocks):
                raise ValueError(
                    f"requested {final_blocks} trainable blocks from a {len(blocks)}-block backbone"
                )
            for block in blocks[-final_blocks:]:
                for parameter in block.parameters():
                    parameter.requires_grad_(True)
            for name in ("conv_head", "bn2"):
                layer = getattr(self.backbone, name, None)
                if layer is not None:
                    for parameter in layer.parameters():
                        parameter.requires_grad_(True)

        @staticmethod
        def _position_basis(slices, device, dtype):
            position = torch.linspace(-1.0, 1.0, slices, device=device, dtype=dtype)
            return torch.stack(
                (
                    position,
                    position.square(),
                    torch.sin(math.pi * position),
                    torch.cos(math.pi * position),
                ),
                dim=-1,
            )

        def forward(self, images, mask):
            batch, planes, slices, channels, height, width = images.shape
            if planes != len(PLANES):
                raise ValueError(f"expected {len(PLANES)} plane slots, got {planes}")
            flat = images.reshape(batch * planes * slices, channels, height, width)
            encoded = []
            train_backbone = self.training and any(
                parameter.requires_grad for parameter in self.backbone.parameters()
            )
            for start in range(0, len(flat), self.backbone_chunk):
                chunk = flat[start : start + self.backbone_chunk]
                if train_backbone:
                    from torch.utils.checkpoint import checkpoint

                    encoded.append(checkpoint(self.backbone, chunk, use_reentrant=False))
                else:
                    encoded.append(self.backbone(chunk))
            features = torch.cat(encoded, dim=0).reshape(batch, planes, slices, -1)

            plane_ids = torch.arange(planes, device=features.device)
            plane_context = self.plane_embedding(plane_ids)[None, :, None, :]
            position = self._position_basis(slices, features.device, features.dtype)
            position_context = self.position_projection(position)[None, None, :, :]
            features = features + plane_context + position_context
            features = features.reshape(batch, planes * slices, -1)
            flat_mask = mask.reshape(batch, planes * slices)

            gated = torch.tanh(self.attn_v(features)) * torch.sigmoid(self.attn_u(features))
            attention = self.attn_out(gated).masked_fill(
                ~flat_mask[:, :, None], -1e4
            ).softmax(dim=1)
            pooled = torch.einsum("bnt,bnf->btf", attention, features)
            attention_logits = (
                torch.einsum("btf,tf->bt", pooled, self.target_weight) + self.target_bias
            )
            max_logits = self.slice_head(features).masked_fill(
                ~flat_mask[:, :, None], -1e4
            ).max(dim=1).values
            mix = torch.sigmoid(self.mix_logit)
            return (1.0 - mix) * attention_logits + mix * max_logits

    return AnatomyAwareKneeMIL


def _make_expert_sampler(rows, fraction, seed, torch):
    """Sample a fixed expected expert fraction without discarding pseudo rows."""
    if not 0.0 < fraction < 1.0:
        raise ValueError("expert_sample_fraction must be strictly between zero and one")
    expert = rows["is_expert"].to_numpy(np.int8).astype(bool)
    n_expert, n_pseudo = int(expert.sum()), int((~expert).sum())
    if n_expert == 0 or n_pseudo == 0:
        raise ValueError("expert-aware sampling requires both expert and pseudo-labelled rows")
    expert_multiplier = fraction * n_pseudo / ((1.0 - fraction) * n_expert)
    weights = np.where(expert, expert_multiplier, 1.0).astype(np.float64)
    generator = torch.Generator().manual_seed(int(seed))
    from torch.utils.data import WeightedRandomSampler

    sampler = WeightedRandomSampler(
        torch.from_numpy(weights),
        num_samples=len(rows),
        replacement=True,
        generator=generator,
    )
    return sampler, {
        "expert_rows": n_expert,
        "pseudo_rows": n_pseudo,
        "expert_multiplier": float(expert_multiplier),
        "expected_expert_fraction": float(fraction),
    }


def train_one_fold(
    label_table,
    series,
    image_root,
    output_dir,
    fold,
    backbone_checkpoint,
    config,
    policy=TrainingPolicy(),
):
    _, _, _, torch, nn, DataLoader, _ = _training_imports()
    random.seed(config.seed + fold)
    np.random.seed(config.seed + fold)
    torch.manual_seed(config.seed + fold)
    Dataset, Model = make_dataset_class(), make_model_class()
    train_rows, valid_rows = BASE.prepare_fold_rows(label_table, fold, config)
    sampler, sampler_meta = _make_expert_sampler(
        train_rows,
        policy.expert_sample_fraction,
        config.seed + 10_007 * fold,
        torch,
    )
    train_loader = DataLoader(
        Dataset(train_rows, series, image_root, config, True, config.seed + fold),
        batch_size=config.batch_size,
        sampler=sampler,
        num_workers=config.workers,
        pin_memory=config.workers > 0,
        persistent_workers=config.workers > 0,
    )
    valid_loader = DataLoader(
        Dataset(valid_rows, series, image_root, config, False, config.seed + fold),
        batch_size=1,
        shuffle=False,
        num_workers=config.workers,
        pin_memory=config.workers > 0,
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("GPU is required")
    core_model = Model(config.backbone, backbone_checkpoint, config.backbone_chunk).to(device)
    model = nn.DataParallel(core_model) if torch.cuda.device_count() > 1 else core_model
    loss_fn = nn.BCEWithLogitsLoss(reduction="none")
    scaler = torch.amp.GradScaler("cuda")
    history = []
    output_dir.mkdir(parents=True, exist_ok=True)
    schedule = (
        [(0, config.frozen_lr)] * config.frozen_epochs
        + [(policy.unfrozen_backbone_blocks, config.unfrozen_lr)] * config.unfrozen_epochs
    )
    optimizer, active_blocks = None, None
    for epoch, (trainable_blocks, learning_rate) in enumerate(schedule):
        if active_blocks != trainable_blocks:
            core_model.set_backbone_stage(trainable_blocks)
            optimizer = torch.optim.AdamW(
                [parameter for parameter in model.parameters() if parameter.requires_grad],
                lr=learning_rate,
                weight_decay=config.weight_decay,
            )
            active_blocks = trainable_blocks
        assert optimizer is not None
        model.train()
        for layer in core_model.backbone.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                layer.eval()
        losses = []
        sampled_expert = 0
        sampled_total = 0
        for step, batch in enumerate(train_loader):
            images, mask = batch["images"].to(device, non_blocking=False), batch["mask"].to(
                device, non_blocking=False
            )
            target, weight = batch["target"].to(device), batch["weight"].to(device)
            # Expert rows have the same gold weight for all twelve targets.
            sampled_expert += int((weight.min(dim=1).values >= config.gold_weight).sum().item())
            sampled_total += int(len(weight))
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                raw = loss_fn(model(images, mask), target)
                loss = (raw * weight).sum() / weight.sum().clamp_min(1.0)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), policy.gradient_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
            # Free activation graphs between steps; T4 OOM risk is real with MIL bags.
            del images, mask, target, weight, raw, loss
            if step % 50 == 0:
                torch.cuda.empty_cache()
        history.append(
            {
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "trainable_backbone_blocks": int(trainable_blocks),
                "learning_rate": float(learning_rate),
                "observed_expert_fraction": sampled_expert / max(sampled_total, 1),
            }
        )
        # Preserve a recoverable checkpoint after every epoch. This is deliberately
        # written before validation, because a resource kill must not erase all
        # evidence from a completed training epoch.
        torch.save(
            {
                "model": core_model.state_dict(),
                "model_family": policy.model_family,
                "backbone": config.backbone,
                "config": asdict(config),
                "training_policy": asdict(policy),
                "sampler": sampler_meta,
                "fold": fold,
                "completed_epoch": epoch,
                "history": history,
                "backbone_checkpoint_sha256": hashlib.sha256(
                    backbone_checkpoint.read_bytes()
                ).hexdigest(),
            },
            output_dir / f"fold{fold}_epoch{epoch}.pt",
        )

    model.eval()
    uids, truth, score = [], [], []
    with torch.no_grad():
        for batch in valid_loader:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(batch["images"].to(device), batch["mask"].to(device))
            uids.extend(batch["uid"])
            truth.append(batch["target"].numpy())
            score.append(torch.sigmoid(logits.float()).cpu().numpy())
    truth, score = np.concatenate(truth), np.concatenate(score)
    macro, per_target = BASE.pooled_macro_auc(truth, score)
    oof = pd.DataFrame({UID: uids})
    valid_meta = valid_rows.set_index(UID)
    oof["fold"] = int(fold)
    oof["is_expert"] = [int(valid_meta.loc[uid, "is_expert"]) for uid in uids]
    for target_index, target in enumerate(TARGETS):
        oof[f"target::{target}"] = truth[:, target_index]
        oof[f"pred::{target}"] = score[:, target_index]
    oof.to_csv(output_dir / f"fold{fold}_full_oof.csv", index=False)
    expert_mask = oof["is_expert"].to_numpy(dtype=bool)
    if expert_mask.any():
        expert_macro, expert_per_target = BASE.pooled_macro_auc(
            truth[expert_mask], score[expert_mask]
        )
    else:
        expert_macro, expert_per_target = float("nan"), {}
    destination = output_dir / f"fold{fold}_final.pt"
    torch.save(
        {
            "model": core_model.state_dict(),
            "model_family": policy.model_family,
            "backbone": config.backbone,
            "config": asdict(config),
            "training_policy": asdict(policy),
            "sampler": sampler_meta,
            "fold": fold,
            "fold_diagnostic_macro_auc": macro,
            "fold_diagnostic_per_target_auc": per_target,
            "fold_expert_macro_auc": expert_macro,
            "fold_expert_per_target_auc": expert_per_target,
            "fold_oof_rows": int(len(oof)),
            "history": history,
            "visible_gpus": torch.cuda.device_count(),
            "backbone_checkpoint_sha256": hashlib.sha256(
                backbone_checkpoint.read_bytes()
            ).hexdigest(),
        },
        destination,
    )
    return destination


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    for name in (
        "train-csv",
        "series-csv",
        "pilkwang-labels",
        "steven-labels",
        "lixin-labels",
        "output-dir",
    ):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--image-root", type=Path)
    parser.add_argument("--backbone-checkpoint", type=Path)
    parser.add_argument("--fold", type=int)
    parser.add_argument("--prepare-only", action="store_true")
    args = parser.parse_args(argv)

    # Expert oversampling replaces some of the multiplicative gold weighting;
    # keeping both at their v1 values would overfit the 46-or-so training experts.
    config = replace(TrainConfig(), gold_weight=2.0)
    policy = TrainingPolicy()
    paths = [
        args.train_csv,
        args.series_csv,
        args.pilkwang_labels,
        args.steven_labels,
        args.lixin_labels,
    ]
    frames = [pd.read_csv(path) for path in paths]
    labels = BASE.build_public_consensus(
        frames[0], frames[2], frames[3], frames[4], config.pseudo_weight_floor
    )
    labels["fold"] = BASE.assign_group_balanced_folds(labels, config.folds, config.seed)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    labels.to_csv(args.output_dir / "fold_labels.csv", index=False)
    manifest = {
        "config": asdict(config),
        "training_policy": asdict(policy),
        "rows": len(labels),
        "expert_rows": int(labels["is_expert"].sum()),
        "fold_counts": labels["fold"].value_counts().sort_index().to_dict(),
        "fold_expert_counts": labels.loc[
            labels["is_expert"].eq(1), "fold"
        ].value_counts().sort_index().to_dict(),
        "inputs": {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths},
        "base_module_sha256": hashlib.sha256(
            Path(BASE.__file__).read_bytes()
        ).hexdigest(),
        "module_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
    }
    (args.output_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n"
    )
    print(json.dumps(manifest, indent=2, sort_keys=True))
    if args.prepare_only:
        return
    if args.fold is None or args.image_root is None or args.backbone_checkpoint is None:
        parser.error("training requires --fold, --image-root, and --backbone-checkpoint")
    if not 0 <= args.fold < config.folds:
        parser.error(f"--fold must be in [0, {config.folds - 1}]")
    print(
        "saved",
        train_one_fold(
            labels,
            frames[1],
            args.image_root,
            args.output_dir,
            args.fold,
            args.backbone_checkpoint,
            config,
            policy,
        ),
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/v3_b3_source/efficientnet_b3_public_repro_v4_t4.py
#!/usr/bin/env python3
"""T4-bounded, no-flip EfficientNet-B3 training for RSNA Knee.

This is an independent reproduction of the public .903 recipe.  It keeps the
audited v2 model and report-label contracts, but corrects two important input
details disclosed by that notebook: non-square slices are padded before resize,
and no horizontal flip is ever applied to laterality-specific targets.  V3 runs
one fold per Kaggle job, allowing the full 288px / 12-slice / eight-epoch recipe
to stay inside the code-competition runtime budget.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import math
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd


def _load_v2():
    path = Path(__file__).with_name("efficientnet_b3_public_repro_v2_anatomy.py")
    if not path.is_file():
        raise FileNotFoundError(f"audited v2 dependency is absent: {path}")
    name = "rsna_knee_effnet_b3_public_repro_v2_t4_dependency"
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import audited v2 dependency: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


V2 = _load_v2()
BASE = V2.BASE
TARGETS, PLANES, UID = V2.TARGETS, V2.PLANES, V2.UID
TrainConfig = V2.TrainConfig
TrainingPolicy = V2.TrainingPolicy
pick_series_slots = V2.pick_series_slots
uniform_slice_indices = V2.uniform_slice_indices
stochastic_slice_indices = V2.stochastic_slice_indices
make_model_class = V2.make_model_class
_training_imports = V2._training_imports
CACHE_ROOT = Path("/kaggle/working/v3_1_uint8_cache")


def _natural_key(path: Path):
    import re

    return tuple(
        int(part) if part.isdigit() else part.casefold()
        for part in re.split(r"(\d+)", path.name)
    )


def load_dicom_volume(series_dir: Path, image_size: int, max_slices: int = 64):
    """Decode like the .903 inference path, including square padding."""
    cv2, pydicom, _, _, _, _, _ = _training_imports()
    slices = []
    for path in sorted(Path(series_dir).glob("*.dcm"), key=_natural_key):
        try:
            ds = pydicom.dcmread(str(path), force=True)
            image = ds.pixel_array.astype(np.float32)
            image *= float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
            image += float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
            if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
                image = float(np.max(image)) - image
            instance = getattr(ds, "InstanceNumber", None)
            try:
                instance = int(instance) if instance is not None else None
            except (TypeError, ValueError):
                instance = None
            location = getattr(ds, "SliceLocation", None)
            if location is None:
                position = getattr(ds, "ImagePositionPatient", None)
                location = position[2] if position is not None and len(position) >= 3 else None
            try:
                location = float(location) if location is not None else None
            except (TypeError, ValueError):
                location = None
            slices.append((instance, location, path.name, image))
        except Exception:
            continue
    if not slices:
        raise RuntimeError(f"no decodable slices in {series_dir}")

    slices.sort(
        key=lambda item: (
            0 if item[0] is not None else 1 if item[1] is not None else 2,
            item[0] if item[0] is not None else 0,
            item[1] if item[1] is not None else 0.0,
            item[2],
        )
    )
    volume = np.stack([item[3] for item in slices]).astype(np.float32, copy=False)
    lo, hi = np.percentile(volume, [1.0, 99.0])
    hi = max(float(hi), float(lo) + 1.0)
    volume = np.clip((volume - float(lo)) / (hi - float(lo)), 0.0, 1.0)
    if len(volume) > int(max_slices):
        start = (len(volume) - int(max_slices)) // 2
        volume = volume[start : start + int(max_slices)]

    resized = []
    for image in volume:
        height, width = image.shape
        if height != width:
            side = max(height, width)
            padded = np.zeros((side, side), dtype=image.dtype)
            top, left = (side - height) // 2, (side - width) // 2
            padded[top : top + height, left : left + width] = image
            image = padded
        resized.append(
            cv2.resize(image, (int(image_size), int(image_size)), interpolation=cv2.INTER_AREA)
        )
    return np.rint(np.stack(resized) * 255.0).astype(np.uint8).astype(np.float32) / 255.0


def make_dataset_class():
    """Fixed-plane dataset with intensity jitter and explicitly no flip."""
    _, _, _, torch, _, _, Dataset = _training_imports()

    class NoFlipKneeStudyDataset(Dataset):
        def __init__(self, rows, series, image_root, config, training, seed):
            self.rows = rows.reset_index(drop=True)
            self.series = {
                str(uid): group.copy() for uid, group in series.groupby(UID, sort=False)
            }
            self.image_root = Path(image_root)
            self.config = config
            self.training = bool(training)
            self.seed = int(seed)
            # Production prebuilt caches live at CACHE_ROOT/studies/{uid}.npz
            # (often under read-only /kaggle/input). On-demand session caches MUST
            # write under /kaggle/working — never mkdir inside input mounts.
            self.prebuilt_studies = CACHE_ROOT / "studies"
            session_base = Path(
                os.environ.get("RSNA_SESSION_CACHE", "/kaggle/working/v3_1_session_cache")
            )
            self.cache_root = session_base / (
                f"{config.image_size}px_{config.train_slices if training else config.valid_slices}s_"
                f"{'train' if training else 'valid'}"
            )
            self.cache_root.mkdir(parents=True, exist_ok=True)

        def __len__(self):
            return len(self.rows)

        def _load_cached(self, path: Path, count: int, image_size: int):
            with np.load(path, allow_pickle=False) as cached:
                raw_planes = cached["images"]
                masks = cached["mask"].astype(bool)
            if raw_planes.ndim != 4:
                raise ValueError(f"unexpected cache rank for {path}: {raw_planes.shape}")
            # Prebuilt tensors are [planes, slices, H, W]. If the recipe requests
            # fewer slices, subsample uniformly; more slices than cached is refused
            # so we never silently stretch a 12-slice cache to 32.
            planes, slices, height, width = raw_planes.shape
            if height != image_size or width != image_size:
                raise ValueError(
                    f"cache spatial size {height}x{width} != config {image_size} for {path}"
                )
            if slices == count:
                return raw_planes, masks
            if slices > count:
                indices = uniform_slice_indices(slices, count)
                return raw_planes[:, indices], masks[:, indices]
            raise ValueError(
                f"cache has {slices} slices but config wants {count} for {path}"
            )

        def __getitem__(self, index):
            row, config = self.rows.iloc[index], self.config
            uid = str(row[UID])
            count = config.train_slices if self.training else config.valid_slices
            prebuilt = self.prebuilt_studies / f"{uid}.npz"
            session_cache = self.cache_root / f"{uid}.npz"
            if prebuilt.is_file():
                raw_planes, masks = self._load_cached(prebuilt, count, config.image_size)
            elif session_cache.is_file():
                raw_planes, masks = self._load_cached(
                    session_cache, count, config.image_size
                )
            else:
                visit_seed = (
                    int(torch.randint(0, 2**31 - 1, (1,)).item())
                    if self.training
                    else 0
                )
                rng = np.random.default_rng(self.seed + 1_000_003 * index + visit_seed)
                empty = pd.DataFrame(
                    columns=[
                        "SeriesInstanceUID",
                        "Anatomical_Plane",
                        "Fluid_Sensitive",
                        "Fat_Suppression",
                    ]
                )
                selected = pick_series_slots(self.series.get(uid, empty), len(PLANES))
                planes, mask_rows = [], []
                for series_uid in selected:
                    if series_uid is None:
                        planes.append(
                            np.zeros((count, config.image_size, config.image_size), np.float32)
                        )
                        mask_rows.append(np.zeros(count, bool))
                        continue
                    volume = load_dicom_volume(
                        self.image_root / uid / str(series_uid),
                        config.image_size,
                        config.max_series_slices,
                    )
                    indices = (
                        stochastic_slice_indices(len(volume), count, rng)
                        if self.training
                        else uniform_slice_indices(len(volume), count)
                    )
                    # Cache stores raw windowed intensities; jitter is applied after
                    # load so prebuilt and on-demand paths share one augmentation site.
                    planes.append(volume[indices].astype(np.float32))
                    mask_rows.append(np.ones(count, bool))
                raw_planes = np.rint(np.stack(planes) * 255.0).astype(np.uint8)
                masks = np.stack(mask_rows)
                np.savez(session_cache, images=raw_planes, mask=masks.astype(np.uint8))
            float_planes = raw_planes.astype(np.float32) / 255.0
            if self.training:
                # Light intensity jitter only; never horizontal flip (laterality).
                rng = np.random.default_rng(self.seed + 17 * index + (index * 1_000_003))
                scale = float(rng.uniform(0.88, 1.12))
                gamma = float(rng.uniform(0.88, 1.12))
                float_planes = np.clip(float_planes * scale, 0.0, 1.0)
                float_planes = np.clip(float_planes ** gamma, 0.0, 1.0)
            planes = (float_planes - 0.449) / 0.226
            return {
                "uid": uid,
                "images": torch.from_numpy(planes[:, :, None]),
                "mask": torch.from_numpy(masks),
                "target": torch.tensor(
                    [row[f"target::{target}"] for target in TARGETS], dtype=torch.float32
                ),
                "weight": torch.tensor(
                    [row[f"train_weight::{target}"] for target in TARGETS],
                    dtype=torch.float32,
                ),
            }

    return NoFlipKneeStudyDataset


# V2.train_one_fold resolves these names from its own module globals.
V2.load_dicom_volume = load_dicom_volume
V2.make_dataset_class = make_dataset_class


def prepare_full_fold_rows(
    label_table: pd.DataFrame, fold: int, config: TrainConfig
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return leakage-safe training rows and every row in the held-out fold."""
    train_rows = label_table.loc[label_table["fold"].ne(fold)].copy()
    valid_rows = label_table.loc[label_table["fold"].eq(fold)].copy()
    if train_rows.empty or valid_rows.empty:
        raise ValueError(f"empty train/validation split for fold {fold}")
    for target in TARGETS:
        train_gold = pd.to_numeric(train_rows[f"gold::{target}"], errors="coerce")
        train_is_gold = train_rows["is_expert"].eq(1) & train_gold.notna()
        train_rows[f"target::{target}"] = train_rows[f"pseudo::{target}"].where(
            ~train_is_gold, train_gold
        )
        train_rows[f"train_weight::{target}"] = train_rows[f"weight::{target}"].where(
            ~train_is_gold, config.gold_weight
        )
        valid_gold = pd.to_numeric(valid_rows[f"gold::{target}"], errors="coerce")
        valid_is_gold = valid_rows["is_expert"].eq(1) & valid_gold.notna()
        valid_rows[f"target::{target}"] = valid_rows[f"pseudo::{target}"].where(
            ~valid_is_gold, valid_gold
        )
        valid_rows[f"train_weight::{target}"] = 1.0
    if set(train_rows["report_group"]).intersection(valid_rows["report_group"]):
        raise AssertionError("normalized report leakage across train and validation")
    return train_rows, valid_rows


V2.BASE.prepare_fold_rows = prepare_full_fold_rows


def load_v3_label_table(
    train: pd.DataFrame, labels: pd.DataFrame, folds: pd.DataFrame
) -> pd.DataFrame:
    """Convert the audited V3 artifacts to the trainer's leakage-safe schema."""
    required_train = {UID, "Report", *TARGETS}
    required_labels = {UID, *TARGETS, *(f"{target}__weight" for target in TARGETS)}
    required_folds = {UID, "fold", "report_group", "is_gold"}
    for name, frame, required in (
        ("train", train, required_train),
        ("labels_v3", labels, required_labels),
        ("folds_v3", folds, required_folds),
    ):
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"{name} is missing columns: {sorted(missing)}")
        if frame[UID].duplicated().any():
            raise ValueError(f"{name} contains duplicate {UID} rows")

    expected = set(train[UID].astype(str))
    for name, frame in (("labels_v3", labels), ("folds_v3", folds)):
        observed = set(frame[UID].astype(str))
        if observed != expected:
            raise ValueError(
                f"{name} UID mismatch: missing={len(expected - observed)}, "
                f"extra={len(observed - expected)}"
            )

    base = train[[UID, *TARGETS]].copy()
    label_columns = [UID]
    for target in TARGETS:
        label_columns.extend([target, f"{target}__weight"])
    out = base.merge(labels[label_columns], on=UID, how="inner", suffixes=("__gold", "__pseudo"))
    out = out.merge(folds[[UID, "fold", "report_group", "is_gold"]], on=UID, how="inner")
    out["fold"] = pd.to_numeric(out["fold"], errors="raise").astype(np.int8)
    out["is_expert"] = pd.to_numeric(out["is_gold"], errors="raise").astype(np.int8)
    if sorted(out["fold"].unique().tolist()) != list(range(5)):
        raise ValueError(f"unexpected folds: {sorted(out['fold'].unique().tolist())}")
    if int(out["is_expert"].sum()) != int(train[list(TARGETS)].notna().all(axis=1).sum()):
        raise ValueError("is_gold does not match the expert labels in train.csv")

    result = out[[UID, "report_group", "is_expert", "fold"]].copy()
    for target in TARGETS:
        pseudo = pd.to_numeric(out[f"{target}__pseudo"], errors="raise")
        weight = pd.to_numeric(out[f"{target}__weight"], errors="raise")
        gold = pd.to_numeric(out[f"{target}__gold"], errors="coerce")
        if not pseudo.between(0.0, 1.0).all():
            raise ValueError(f"V3 pseudo-labels outside [0, 1] for {target}")
        if not weight.between(0.0, 3.0, inclusive="both").all():
            raise ValueError(f"V3 weights outside [0, 3] for {target}")
        result[f"pseudo::{target}"] = pseudo.astype(np.float32)
        result[f"weight::{target}"] = weight.astype(np.float32)
        result[f"gold::{target}"] = gold
    return result


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    for name in (
        "train-csv",
        "series-csv",
        "v3-labels",
        "v3-folds",
        "output-dir",
    ):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--image-root", type=Path)
    parser.add_argument("--backbone-checkpoint", type=Path)
    parser.add_argument("--fold", type=int)
    parser.add_argument("--prepare-only", action="store_true")
    args = parser.parse_args(argv)

    config = replace(
        TrainConfig(),
        image_size=224,
        train_slices=8,
        valid_slices=8,
        batch_size=2,
        workers=0,
        backbone_chunk=8,
        frozen_epochs=1,
        unfrozen_epochs=0,
        frozen_lr=8e-4,
        unfrozen_lr=8e-5,
        gold_weight=2.0,
        backbone="efficientnet_b3",
    )
    policy = replace(
        TrainingPolicy(),
        expert_sample_fraction=0.15,
        unfrozen_backbone_blocks=3,
        model_family="efficientnet_b3_noflip_v3_1_smoke",
    )
    paths = [
        args.train_csv,
        args.series_csv,
        args.v3_labels,
        args.v3_folds,
    ]
    frames = [pd.read_csv(path) for path in paths]
    labels = load_v3_label_table(frames[0], frames[2], frames[3])
    args.output_dir.mkdir(parents=True, exist_ok=True)
    labels.to_csv(args.output_dir / "fold_labels.csv", index=False)
    manifest = {
        "status": "PREPARED",
        "config": asdict(config),
        "training_policy": asdict(policy),
        "rows": len(labels),
        "expert_rows": int(labels["is_expert"].sum()),
        "fold_counts": labels["fold"].value_counts().sort_index().to_dict(),
        "fold_expert_counts": labels.loc[
            labels["is_expert"].eq(1), "fold"
        ].value_counts().sort_index().to_dict(),
        "inputs": {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths},
        "module_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        "no_horizontal_flip": True,
        "square_padding_before_resize": True,
        "fold_cross_fitted_labels": True,
        "full_heldout_fold_oof": True,
        "uint8_disk_cache": str(CACHE_ROOT),
    }
    (args.output_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n"
    )
    print(json.dumps(manifest, indent=2, sort_keys=True), flush=True)
    if args.prepare_only:
        return
    if args.fold is None or args.image_root is None or args.backbone_checkpoint is None:
        parser.error("training requires --fold, --image-root, and --backbone-checkpoint")
    if not 0 <= args.fold < config.folds:
        parser.error(f"--fold must be in [0, {config.folds - 1}]")
    print(
        "saved",
        V2.train_one_fold(
            labels,
            frames[1],
            args.image_root,
            args.output_dir,
            args.fold,
            args.backbone_checkpoint,
            config,
            policy,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/v3_b3_source/efficientnet_b3_public_repro_v5_full.py
#!/usr/bin/env python3
"""Production EfficientNet-B3 fold trainer for RSNA Knee top-50 / top-10 path.

Uses the audited v2 anatomy model + v4 no-flip / square-pad contracts, but:
  * full 288px / 12-slice recipe (not the 224 smoke)
  * fold-cross-fitted V3.1 labels (labels_v3_fold{N}.csv)
  * optional prebuilt uint8 cache (mounted Kaggle dataset or kernel output)
  * epoch checkpoints + full held-out-fold OOF
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import os
import sys
from dataclasses import asdict, replace
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd


def _load_v4():
    path = Path(__file__).with_name("efficientnet_b3_public_repro_v4_t4.py")
    if not path.is_file():
        raise FileNotFoundError(f"v4 dependency missing: {path}")
    name = "rsna_knee_effnet_b3_v4_dependency"
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


V4 = _load_v4()
V2 = V4.V2
BASE = V4.BASE
TARGETS, PLANES, UID = V4.TARGETS, V4.PLANES, V4.UID
TrainConfig = V4.TrainConfig
TrainingPolicy = V4.TrainingPolicy
load_v3_label_table = V4.load_v3_label_table
prepare_full_fold_rows = V4.prepare_full_fold_rows


def resolve_cache_root(explicit: Optional[Path]) -> Path:
    """Prefer an attached complete cache, else working directory."""
    candidates: list[Path] = []
    if explicit is not None:
        candidates.append(explicit)
    env = os.environ.get("RSNA_CACHE_ROOT", "").strip()
    if env:
        candidates.append(Path(env))

    root = Path("/kaggle/input")
    if root.is_dir():
        for manifest in root.rglob("cache_manifest.json"):
            candidates.append(manifest.parent)
        for studies in root.rglob("studies"):
            if studies.is_dir() and any(studies.glob("*.npz")):
                candidates.append(studies.parent)

    candidates.extend(
        [
            Path("/kaggle/working/v3_1_cache"),
            Path("/kaggle/working/v3_1_uint8_cache"),
        ]
    )

    best: Optional[Path] = None
    best_count = -1
    for candidate in candidates:
        if candidate is None:
            continue
        studies = candidate / "studies"
        if not studies.is_dir():
            continue
        count = sum(1 for _ in studies.glob("*.npz"))
        if count > best_count:
            best, best_count = candidate, count
    if best is not None and best_count > 0:
        return best
    return Path("/kaggle/working/v3_1_uint8_cache")


def patch_cache_root(cache_root: Path) -> None:
    """Force v4 dataset class to read the production cache."""
    V4.CACHE_ROOT = cache_root
    # v4 make_dataset_class closes over CACHE_ROOT at call time via module global.
    # Re-bind and rebuild dataset factory.
    original = V4.make_dataset_class

    def make_dataset_class():
        # Temporarily ensure CACHE_ROOT is current
        V4.CACHE_ROOT = cache_root
        return original()

    V4.make_dataset_class = make_dataset_class
    V2.make_dataset_class = make_dataset_class
    V2.load_dicom_volume = V4.load_dicom_volume


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("train-csv", "series-csv", "v3-labels", "v3-folds", "output-dir"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--image-root", type=Path)
    parser.add_argument("--backbone-checkpoint", type=Path)
    parser.add_argument("--fold", type=int)
    parser.add_argument("--cache-root", type=Path)
    parser.add_argument("--image-size", type=int, default=288)
    parser.add_argument("--train-slices", type=int, default=8)
    parser.add_argument("--valid-slices", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--backbone-chunk", type=int, default=6)
    parser.add_argument("--frozen-epochs", type=int, default=1)
    # Wave-1 after OOM: 1+3 epochs, batch1, 8 slices (docs/gpu-budget-28h.md).
    # Wave-2 / V6: prefer 1+7 to approach the public 8-epoch recipe without OOM.
    parser.add_argument("--unfrozen-epochs", type=int, default=3)
    parser.add_argument("--gold-weight", type=float, default=2.0)
    parser.add_argument("--expert-sample-fraction", type=float, default=0.15)
    parser.add_argument("--unfrozen-backbone-blocks", type=int, default=3)
    parser.add_argument(
        "--model-family",
        type=str,
        default="efficientnet_b3_noflip_v5_wave1",
    )
    parser.add_argument("--prepare-only", action="store_true")
    parser.add_argument("--require-cache", action="store_true")
    args = parser.parse_args(argv)

    train_started = __import__("time").time()
    cache_root = resolve_cache_root(args.cache_root)
    patch_cache_root(cache_root)

    config = replace(
        TrainConfig(),
        image_size=int(args.image_size),
        train_slices=int(args.train_slices),
        valid_slices=int(args.valid_slices),
        batch_size=int(args.batch_size),
        workers=int(args.workers),
        backbone_chunk=int(args.backbone_chunk),
        frozen_epochs=int(args.frozen_epochs),
        unfrozen_epochs=int(args.unfrozen_epochs),
        frozen_lr=8e-4,
        unfrozen_lr=8e-5,
        gold_weight=float(args.gold_weight),
        backbone="efficientnet_b3",
    )
    policy = replace(
        TrainingPolicy(),
        expert_sample_fraction=float(args.expert_sample_fraction),
        unfrozen_backbone_blocks=int(args.unfrozen_backbone_blocks),
        model_family=str(args.model_family),
    )

    # Ensure prepare_fold_rows is the leakage-safe full-fold variant.
    V2.BASE.prepare_fold_rows = prepare_full_fold_rows
    BASE.prepare_fold_rows = prepare_full_fold_rows

    paths = [args.train_csv, args.series_csv, args.v3_labels, args.v3_folds]
    frames = [pd.read_csv(path) for path in paths]
    labels = load_v3_label_table(frames[0], frames[2], frames[3])
    args.output_dir.mkdir(parents=True, exist_ok=True)
    labels.to_csv(args.output_dir / "fold_labels.csv", index=False)

    cache_files = list((cache_root / "studies").glob("*.npz")) if (cache_root / "studies").is_dir() else []
    manifest = {
        "status": "PREPARED",
        "config": asdict(config),
        "training_policy": asdict(policy),
        "rows": len(labels),
        "expert_rows": int(labels["is_expert"].sum()),
        "fold_counts": labels["fold"].value_counts().sort_index().to_dict(),
        "fold_expert_counts": labels.loc[labels["is_expert"].eq(1), "fold"]
        .value_counts()
        .sort_index()
        .to_dict(),
        "inputs": {
            str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths
        },
        "module_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        "no_horizontal_flip": True,
        "square_padding_before_resize": True,
        "fold_cross_fitted_labels": True,
        "full_heldout_fold_oof": True,
        "cache_root": str(cache_root),
        "cache_study_files": len(cache_files),
        "require_cache": bool(args.require_cache),
        "gpu_budget_note": "wave1_1plus4_epochs_28h_plan",
    }
    (args.output_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n"
    )
    print(json.dumps(manifest, indent=2, sort_keys=True), flush=True)

    if args.require_cache and len(cache_files) < 4000:
        raise RuntimeError(
            f"require-cache set but only {len(cache_files)} study caches under {cache_root}"
        )

    if args.prepare_only:
        return
    if args.fold is None or args.image_root is None or args.backbone_checkpoint is None:
        parser.error("training requires --fold, --image-root, and --backbone-checkpoint")
    if not 0 <= args.fold < config.folds:
        parser.error(f"--fold must be in [0, {config.folds - 1}]")

    destination = V2.train_one_fold(
        labels,
        frames[1],
        args.image_root,
        args.output_dir,
        args.fold,
        args.backbone_checkpoint,
        config,
        policy,
    )
    elapsed = __import__("time").time() - train_started
    timing = {
        "fold": int(args.fold),
        "elapsed_seconds": elapsed,
        "elapsed_hours": elapsed / 3600.0,
        "epochs_total": int(args.frozen_epochs) + int(args.unfrozen_epochs),
        "checkpoint": str(destination),
        "projected_five_fold_hours": 5.0 * (elapsed / 3600.0),
        "gpu_budget_remaining_hint": "stop_if_five_fold_projection_gt_20h",
    }
    (args.output_dir / f"fold{args.fold}_timing.json").write_text(
        json.dumps(timing, indent=2, sort_keys=True) + "\n"
    )
    print("saved", destination, flush=True)
    print(json.dumps(timing, indent=2, sort_keys=True), flush=True)
    if timing["projected_five_fold_hours"] > 20.0:
        print(
            "WARNING: five-fold projection exceeds 20 GPU hours; "
            "reduce unfrozen-epochs for remaining folds.",
            flush=True,
        )


if __name__ == "__main__":
    main()


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

competition = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
if not competition.is_dir():
    matches = list(Path("/kaggle/input").glob("**/train_series.csv"))
    if not matches:
        raise FileNotFoundError("competition train_series.csv unavailable")
    competition = matches[0].parent

def unique_file(pattern):
    matches = sorted(Path("/kaggle/input").glob(pattern))
    if len(matches) != 1:
        preferred = [m for m in matches if "rsna-knee-v3-labels" in str(m) or "labels_v3" in str(m)]
        if len(preferred) == 1:
            return preferred[0]
        # Prefer exact fold label basename match
        exact = [m for m in matches if m.name == pattern.split("/")[-1]]
        if len(exact) == 1:
            return exact[0]
        raise RuntimeError(f"expected one {pattern}, found {matches}")
    return matches[0]

labels = unique_file("**/labels_v3_fold1.csv")
folds = unique_file("**/folds_v3.csv")
checkpoint = unique_file("**/tf_efficientnet_b3*.pth")
output = Path("/kaggle/working/v3_b3_fold1")
command = [
    sys.executable,
    "/kaggle/working/v3_b3_source/efficientnet_b3_public_repro_v5_full.py",
    "--train-csv", str(competition / "train.csv"),
    "--series-csv", str(competition / "train_series.csv"),
    "--v3-labels", str(labels),
    "--v3-folds", str(folds),
    "--image-root", str(competition / "train_series"),
    "--backbone-checkpoint", str(checkpoint),
    "--output-dir", str(output),
    "--fold", "1",
    "--image-size", "288",
    "--train-slices", "8",
    "--valid-slices", "8",
    "--batch-size", "1",
    "--workers", "0",
    "--backbone-chunk", "6",
    "--frozen-epochs", "1",
    "--unfrozen-epochs", "7",
    "--gold-weight", "3.0",
    "--expert-sample-fraction", "0.20",
    "--unfrozen-backbone-blocks", "4",
    "--model-family", "efficientnet_b3_noflip_v6_long",
    "--require-cache",
]
print("V6 Launching:", " ".join(command), flush=True)
subprocess.run(command, check=True)
print("artifacts:", sorted(p.name for p in output.glob("*")), flush=True)
